## MIS373 - AI For Business - Assignment

**Student Name:** Dion Weerapperuma

**Student ID:** 219273645

## Table of Content
   
## Table of Content

1. [Executive Summary](#cell_executivesummary)


2. [Data Exploration](#cell_exploration)


3. [Sentiment Analysis](#cell_sentiment)


4. [Topic Modeling](#cell_TopicModeling)


<a id = "cell_executivesummary"></a>
### 1. Executive Summary


In this document, we will investigate the data gathered from Twitter (X) by extracting information using data analysis techniques. Within the data analysis, Social Analytics will be able to help marketers gain a clearer understanding of user sentiment as well as popular topics of conversation. By extracting popular hashtags within the dataset and listing the number of it's mentions, we were able to decipher these different topics which were accompanied with visualisations to support the data. Furthermore, the top active users were located which may be beneficial for organisations to be able to track influential individuals that have the ability to attract a range of consumers as well as gain insightful feedback regarding their product. Additionally, by analysing the tweets with the highest number of retweets, marketers and organisations are able to justify what topic grabs the most attention amongst the 50,000 tweets. One approach that was used was to identify the sentiments expressed regarding OpenAI, Google and Microsoft were shared across the platform which was further analysed to depict which company received more positive sentiments. By utilising certain data techniques, analysts can then help pinpoint strengths and weaknesses of the business through the understanding of consumer sentiments. Finally, the second approach used was aimed to look into commonly mentioned topics providing organisations with insights into user interests that can be beneficial towards product development and marketing strategies.


<a id = "cell_exploration"></a>
### 2. Data Exploration



In [ ]:
#(A)
import pandas as pd

df = pd.read_csv('chatGPT-Twitter.csv')
df.head()

In [ ]:
import re

from collections import Counter # To count the number of times events occur

In [ ]:
# Define function
def extract_hashtags(Text):
    hashtags = []
    words = Text.split()
    for word in words:
        if word.startswith("#"):
            clean_hashtag = re.sub(r'[^a-zA-Z0-9_]', '', word[1:])  # Remove special characters
            if clean_hashtag:  # Only include non-empty hashtags
                hashtags.append("#" + clean_hashtag.lower())  # Convert to lowercase
    return hashtags

# Extract function
df['hashtags'] = df['Text'].apply(extract_hashtags)

all_hashtags = [hashtag for sublist in df['hashtags'] for hashtag in sublist]
hashtag_counts = Counter(all_hashtags)

# Remove '#chatgpt'
hashtag_counts.pop("#chatgpt", None)

# Get the top 10 most popular hashtags + mentions
top_hashtags = hashtag_counts.most_common(10)

print("Top hashtags mentioned in Text")
for hashtag, count in top_hashtags:
    print(f"{hashtag}: {count} mentions") # List the number of occurrences

In [ ]:
import matplotlib.pyplot as plt # Library to create visualisations

# Extract hashtags
hashtags, frequencies = zip(*top_hashtags)

# Plot bar chart
plt.figure(figsize=(10, 7))
plt.bar(hashtags, frequencies, color='darkblue')
plt.xlabel('Hashtags')
plt.ylabel('Mentions')
plt.title('Top 10 Hashtags mentioned')
plt.xticks(rotation=45)
plt.tight_layout()

In [ ]:
#(B)
top_users = df['Username'].value_counts().head(10)

print("Top 10 most active users tweeting about ChatGPT:")
print(top_users)

In [ ]:
# Plot the bar chart
plt.figure(figsize=(10, 7))
top_users.plot(kind='bar', color='darkred')
plt.xlabel('Usernames')
plt.ylabel('Number of Tweets')
plt.title('Top 10 Most Active Users Tweeting About ChatGPT')
plt.xticks(rotation=45)
plt.tight_layout()

In [ ]:
#(C)
retweet_df = df.sort_values(by='RetweetCount', ascending=False) # Define retweet_df
tweets = retweet_df.head(6) # Top 6 tweets with most retweets

print("Top 6 tweets with the most retweets:") # Display tweets
for index, tweet in tweets.iterrows():
    print(tweet['Text'])
    print("Number of retweets:", tweet['RetweetCount'])
    print()


<a id = "cell_sentiment"></a>
### 3. Sentiment Analysis



In [ ]:
#(D)
import nltk
nltk.download('vader_lexicon')
from nltk.sentiment.vader import SentimentIntensityAnalyzer

In [ ]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer

nltk.download('punkt')
nltk.download('stopwords')

# Tokenization, noise removal, and normalization
def preprocess_text(text):

    tokens = word_tokenize(text)

    stop_words = set(stopwords.words('english'))
    tokens = [token.lower() for token in tokens if token not in stop_words and token.isalpha()]

    preprocessed_text = ' '.join(tokens)

    return preprocessed_text

# Apply preprocessing to the Text column
df['Preprocessed Text'] = df['Text'].apply(preprocess_text)

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df['Preprocessed Text'])

tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=vectorizer.get_feature_names_out())

In [ ]:
sid = SentimentIntensityAnalyzer()

message_text = df['Text'][1]
print('Review Comment:\n', message_text)

In [ ]:
openai_tweets = df[df['Text'].str.contains('OpenAI', case=False)].copy()

# Display Tweets mentioning OpenAI
print("Messages mentioning OpenAI:")
for Text in openai_tweets['Text'].tail():
    print(Text)


In [ ]:
# Polarity Scores
scores = sid.polarity_scores(message_text)
for key in sorted(scores):
  print('{0}: {1} \n'.format(key, scores[key]), end='')
print('Score of Tweet: ', df['Text'][5])

In [ ]:
def classify_sentiment(tweet):
    scores = sid.polarity_scores(tweet)
    if scores['compound'] >= 0.05:
        return 'Positive'
    elif scores['compound'] <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

# Sentiment for Tweets about OpenAI
openai_tweets['Sentiment'] = openai_tweets['Text'].apply(classify_sentiment)

# Polarity Score for Tweet
openai_tweets['Compound Score'] = openai_tweets['Text'].apply(lambda x: sid.polarity_scores(x)['compound'])

# Count per Sentiment
sentiment_counts = openai_tweets['Sentiment'].value_counts()

# Plot the sentiment distribution
plt.bar(sentiment_counts.index, sentiment_counts.values, color=['green', 'red', 'blue'])
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.title('Sentiment Distribution of Tweets about OpenAI')

compound_summary = openai_tweets.groupby('Sentiment')['Compound Score'].describe()
print(compound_summary)

<a id = "cell_TopicModeling"></a>
### 4. Topic Modeling



In [ ]:
#(E)
from nltk.stem import PorterStemmer

porter = PorterStemmer()

documents = df['Text']
Cleaned_doc = []
for r in range(len(documents)):
    review = documents[r]
    try:
        review = re.sub('[^A-Za-z]', ' ', review) # Removal of special characters
        review = review.lower() # Convert to lowercase
        Tokens = review.split() #Tokenization

        Filtered_token = [w for w in Tokens if len(w)>3]
        review = ' '.join(Filtered_token)
    except:
        continue
    Cleaned_doc.append(review)
if r < 5:
    print('-[Review Text]: ', review)

In [ ]:
stop_words = stopwords.words('english')

# Remove Stop Words
for r in range(len(Cleaned_doc)):
    each_item = []
    for t in Cleaned_doc[r].split():
        if t not in stop_words:
             each_item.append(t)
    Cleaned_doc[r] = ' '.join(each_item)
    if r < 10:
       print('-[Cleaned Text]: ', Cleaned_doc[r])

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

count_vectorizer = CountVectorizer()

count_data = count_vectorizer.fit_transform(Cleaned_doc)
count_data

In [ ]:
import numpy as np
terms = count_vectorizer.get_feature_names_out()
total_counts = np.zeros_like(count_data[0].toarray()[0])

# Count the popularity of words

for t in count_data:
    total_counts+=t.toarray()[0]

count_dict = (zip(terms, total_counts))
count_dict = sorted(count_dict, key=lambda x:x[1], reverse=True)[0:20] #Take the top 20 words

words = [w[0] for w in count_dict]
counts = [w[1] for w in count_dict]
x_pos = np.arange(len(words))

plt.figure(2, figsize=(11, 3))
plt.subplot(title='20 most common words')
plt.bar(words, counts)
plt.xticks(x_pos, words, rotation=40)
plt.xlabel('words')
plt.ylabel('counts')
plt.show()

In [ ]:
# Find count less than 3500 and above 10
keepIndex = [];
for r in range(len(total_counts)):
    if total_counts[r] < 3500 and total_counts[r] > 10:
        keepIndex.append(r)

print('Number of Terms Remained: ', len(keepIndex))


ReducedTerm = [terms[r] for r in keepIndex]
ReducedCount = count_data[:,keepIndex]
ReducedCount

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation as LDA

# Adjust the two parameters
number_topics = 10

lda = LDA(n_components=number_topics, n_jobs=-1, random_state=2024)
lda.fit(ReducedCount)
#Trained LDA model
lda.components_

Three topics were selected for analysis: OpenAI, Google and Artificial.

OpenAI was chosen because understanding public perception of the organisation behind ChatGPT is just as valuable as analysing the product itself. Sentiment around OpenAI reflects how much credit and trust the public attributes to the developer, not just the tool.

Google was included as a point of comparison. As the most widely used search engine and an established name in technology, how people discuss Google alongside ChatGPT gives useful context around how a new AI product is being positioned against an industry giant.

The term Artificial was selected to capture the broader conversation around AI as a whole. Rather than focusing solely on specific companies, tracking this term shows how general public sentiment toward artificial intelligence is shifting as it becomes more present in everyday life.

### References:

Text Generated by ChatGPT, OpenAI, https://chat.openai.com/c/a5db9a9a-3244-4711-b6c8-5c900ea6ad29 accessed on 10 April 2024

Stack Overflow, https://stackoverflow.com/questions/37224419/import-nltk-does-not-work accessed on 10 April 2024

MIS373W03 Lecture, Deakin University, https://d2l.deakin.edu.au/d2l/le/content/1423550/viewContent/7181532/View accessed on 8 April 2024